# Material + Flux Spectrum - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC material transmutation results using the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Method**: Direct `material.transmute()` with multigroup flux (no geometry/transport)
- **Chain**: `chain_endf_b8.1_sfr.xml` (reduced per nuclide, level 4)
- **Energy groups**: 3-group [1e-5, 0.625, 1e5, 2e7] eV
- **Multigroup flux**: [1e12, 5e12, 1e14] n/cm²/s
- **Timesteps**: 3 × 1 hour irradiation + 2 × 15 day cooling
- **Comparison**: Nuclide atom densities at each timestep

In [ ]:
# Setup and imports
import time
import numpy as np
import pandas as pd
import glob
import os
import yamc
import yamc as mc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
mc.transport_data = NUCLEAR_DATA

# Codes/backends to run. Comma-separated list via the MC_BACKENDS env var.
# Tokens:
#   openmc-cpu  - reference OpenMC depletion
#   yamc-cpu    - YAMC transmutation on CPU
#   yamc-gpu    - YAMC transmutation on GPU
# This notebook does a single YAMC run (no cpu/gpu split), so the tokens
# collapse to two booleans: RUN_OPENMC and RUN_YAMC. Any yamc-* token enables
# the YAMC run. Default runs everything. Examples:
#   MC_BACKENDS=openmc-cpu     # OpenMC only
#   MC_BACKENDS=yamc-cpu       # YAMC only
_VALID_BACKENDS = ("openmc-cpu", "yamc-cpu", "yamc-gpu")
_raw = os.environ.get("MC_BACKENDS", "openmc-cpu,yamc-cpu,yamc-gpu")
MC_BACKENDS = [b.strip().lower() for b in _raw.split(",") if b.strip()]
_bad = [b for b in MC_BACKENDS if b not in _VALID_BACKENDS]
if _bad:
    raise ValueError(
        f"MC_BACKENDS contains unknown token(s) {_bad}; "
        f"valid tokens are {list(_VALID_BACKENDS)}"
    )
if not MC_BACKENDS:
    raise ValueError("MC_BACKENDS resolved to an empty list")
RUN_OPENMC = "openmc-cpu" in MC_BACKENDS
RUN_YAMC = any(b.startswith("yamc-") for b in MC_BACKENDS)
print(f"Backends: {MC_BACKENDS}")

# Transmutation data: OpenMC reads the XML chain; YAMC reads the cached arrow
# directory (auto-downloaded into ~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow/).
# The primary chain path matches the transmutation siblings; if it is absent we
# fall back to any openmc_data copy of the same SFR chain so the OpenMC deplete
# solve can run.
_CHAIN_CANDIDATES = [
    "/home/jon/yamc-org/openmc_data/src/openmc_data/depletion/chain_endf_b8.1_sfr.xml",
]
try:
    import openmc_data as _omc_data
    _CHAIN_CANDIDATES.append(
        os.path.join(os.path.dirname(_omc_data.__file__), "depletion", "chain_endf_b8.1_sfr.xml")
    )
except Exception:
    pass
# Any installed openmc_data package copy on this machine.
_CHAIN_CANDIDATES += sorted(glob.glob(
    "/home/jon/*/.venv/lib/python*/site-packages/openmc_data/depletion/chain_endf_b8.1_sfr.xml"
))
CHAIN_FILE_OPENMC = next((p for p in _CHAIN_CANDIDATES if os.path.exists(p)), _CHAIN_CANDIDATES[0])
TRANSMUTATION_DATA = os.path.expanduser("~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow")

# Load the YAMC chain once; reduce(...) returns a fresh Chain per call,
# so the loaded object can be reused across nuclides without mutation.
_t0 = time.perf_counter()
YAMC_CHAIN = mc.TransmutationChain(TRANSMUTATION_DATA)
YAMC_CHAIN_LOAD_S = time.perf_counter() - _t0
print(f"YAMC chain load: {YAMC_CHAIN_LOAD_S:.3f}s ({len(YAMC_CHAIN)} nuclides)")

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Transmutation parameters
TRANSMUTE_ENERGY_GROUPS = [1e-5, 0.625, 1e5, 2e7]
TRANSMUTE_MULTIGROUP_FLUX = [1e12, 5e12, 1e14]
TRANSMUTE_TIMESTEPS = [3600.0, 3600.0, 3600.0, 15 * 86400.0, 15 * 86400.0]
TRANSMUTE_YAMC_SOURCE_RATES = [1.0, 1.0, 1.0, 0.0, 0.0]
TRANSMUTE_OPENMC_SOURCE_RATES = [1.06e14, 1.06e14, 1.06e14, 0.0, 0.0]
TRANSMUTE_PHASES = ["irrad", "irrad", "irrad", "cool", "cool"]
TRANSMUTE_REDUCE_LEVEL = 4

# Density threshold for comparison
DENSITY_THRESHOLD = 1e-15

In [ ]:
# OpenMC results cache - load reference results from disk if a previous
# run produced them with the same parameters; otherwise run OpenMC and
# write the results out for next time. Cached `reduce_s` / `deplete_s`
# are the timings recorded the first time the entry was produced.
import sys
from pathlib import Path as _Path

_repo_root = _Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "cache_utils.py").exists():
    _repo_root = _repo_root.parent
if (_repo_root / "cache_utils.py").exists() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from cache_utils import cached_or_run

CACHE_DIR = _Path("cache")
OPENMC_CACHE_META = {
    "library": NUCLEAR_DATA,
    "chain_file": CHAIN_FILE_OPENMC,
    "energy_groups": list(TRANSMUTE_ENERGY_GROUPS),
    "multigroup_flux": list(TRANSMUTE_MULTIGROUP_FLUX),
    "timesteps": list(TRANSMUTE_TIMESTEPS),
    "openmc_source_rates": list(TRANSMUTE_OPENMC_SOURCE_RATES),
    "reduce_level": TRANSMUTE_REDUCE_LEVEL,
}

In [ ]:
def run_openmc_transmutation(nuclide, reduced_chain, reduce_s):
    """Run OpenMC material.deplete() for a single nuclide.

    ``reduced_chain`` is the OpenMC ``Chain`` object reduced to the same
    level as the YAMC chain so the comparison is symmetric. The matching
    ``reduce_s`` cost (measured by the caller in cell 6) is returned for
    parity with the YAMC timing breakdown.

    Returns dict with reduce_s, deplete_s, time_s and steps.
    """
    import openmc
    import openmc.deplete

    openmc.config['cross_sections'] = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'cross_sections.xml')

    material = openmc.Material(name=nuclide)
    material.add_nuclide(nuclide, 1.0, percent_type='ao')
    material.set_density('g/cm3', 1.0)
    material.volume = 1.0
    material.temperature = 294.0
    material.depletable = True

    t0 = time.perf_counter()
    results = material.deplete(
        multigroup_flux=TRANSMUTE_MULTIGROUP_FLUX,
        energy_group_structure=TRANSMUTE_ENERGY_GROUPS,
        timesteps=TRANSMUTE_TIMESTEPS,
        source_rates=TRANSMUTE_OPENMC_SOURCE_RATES,
        timestep_units='s',
        chain_file=reduced_chain,
    )
    deplete_s = time.perf_counter() - t0

    # OpenMC returns n+1 materials (index 0 = initial state); skip index 0
    steps = []
    for mat in results[1:]:
        steps.append(mat.get_nuclide_atom_densities())

    return {
        'reduce_s': reduce_s,
        'deplete_s': deplete_s,
        'time_s': reduce_s + deplete_s,
        'steps': steps,
    }

In [ ]:
def run_yamc_transmutation(nuclide, neutron_path, reduced_chain):
    """Run YAMC material.transmute() for a single nuclide.

    Reduces the YAMC transmutation chain to the same level as OpenMC so only
    the reachable products need cross-section data. The 3-group flux shape is
    carried as a Histogram source; the Pulse rate is the total flux magnitude
    [n/cm^2/s] (sum of the multigroup flux), matching the OpenMC source_rates.
    Cooling steps (rate 0) become Cooldown entries.

    Returns dict with reduce_s, deplete_s, time_s and steps.
    """
    # Reduce the pre-loaded YAMC chain to this nuclide's neighborhood.
    t0 = time.perf_counter()
    yamc_reduced = YAMC_CHAIN.reduce([nuclide], level=TRANSMUTE_REDUCE_LEVEL)
    reduce_s = time.perf_counter() - t0

    material = mc.Material(
        composition={nuclide: 1.0},
        density=1.0,
        temperature=294,
        volume=1.0,
        transmutable=True,
        name=nuclide,
    )
    material.read_nuclear_data(NUCLEAR_DATA)

    # Spectrum shape as a Histogram; pulse rate carries the absolute magnitude.
    spectrum = mc.NeutronSource(
        energy=mc.sources.Histogram(TRANSMUTE_ENERGY_GROUPS, TRANSMUTE_MULTIGROUP_FLUX)
    )
    flux_magnitude = sum(TRANSMUTE_MULTIGROUP_FLUX)
    schedule_steps = []
    for dt, rate in zip(TRANSMUTE_TIMESTEPS, TRANSMUTE_YAMC_SOURCE_RATES):
        if rate > 0:
            schedule_steps.append(mc.Pulse(source=spectrum, rate=flux_magnitude * rate, duration=dt))
        else:
            schedule_steps.append(mc.Cooldown(duration=dt))
    schedule = mc.PulseSchedule(schedule_steps)

    t1 = time.perf_counter()
    results = material.transmute(schedule=schedule, transmutation_file=yamc_reduced)
    deplete_s = time.perf_counter() - t1

    steps = []
    for mat in results:
        steps.append(dict(mat.nuclides))

    return {
        'reduce_s': reduce_s,
        'deplete_s': deplete_s,
        'time_s': reduce_s + deplete_s,
        'steps': steps,
    }


In [ ]:
def transmute_quality_label(max_rel_diff):
    """Assign quality label based on max relative difference (%) for transmutation."""
    if max_rel_diff < 0.1:
        return "Excellent"
    elif max_rel_diff < 1.0:
        return "Good"
    elif max_rel_diff < 5.0:
        return "Fair"
    else:
        return "Poor"


def compare_transmutation_steps(yamc_result, openmc_result):
    """Compare transmutation results step-by-step.

    Returns dict with detail_rows, summary metrics, and quality.
    """
    detail_rows = []
    shared_rel_diffs = []
    yamc_only_count = 0
    openmc_only_count = 0

    for step_idx in range(len(TRANSMUTE_TIMESTEPS)):
        ymc_step = yamc_result['steps'][step_idx] if step_idx < len(yamc_result['steps']) else {}
        omc_step = openmc_result['steps'][step_idx] if step_idx < len(openmc_result['steps']) else {}

        all_products = set(ymc_step.keys()) | set(omc_step.keys())

        for product in sorted(all_products):
            y = ymc_step.get(product, 0.0)
            o = omc_step.get(product, 0.0)
            max_val = max(abs(y), abs(o))
            if max_val < DENSITY_THRESHOLD:
                continue

            y_above = abs(y) >= DENSITY_THRESHOLD
            o_above = abs(o) >= DENSITY_THRESHOLD
            shared = y_above and o_above

            if max_val > 0:
                rd = abs(y - o) / max_val * 100.0
            else:
                rd = 0.0

            detail_rows.append({
                'product': product,
                'step': step_idx + 1,
                'phase': TRANSMUTE_PHASES[step_idx],
                'yamc': y,
                'openmc': o,
                'rel_diff_pct': rd,
                'shared': shared,
            })

            if shared:
                shared_rel_diffs.append(rd)
            elif y_above and not o_above:
                yamc_only_count += 1
            else:
                openmc_only_count += 1

    if shared_rel_diffs:
        max_rel_diff_pct = max(shared_rel_diffs)
        rms_rel_diff_pct = float(np.sqrt(np.mean(np.array(shared_rel_diffs) ** 2)))
    else:
        max_rel_diff_pct = 0.0
        rms_rel_diff_pct = 0.0

    n_shared = len({dr['product'] for dr in detail_rows if dr['shared']})
    n_total = len({dr['product'] for dr in detail_rows})

    quality = transmute_quality_label(max_rel_diff_pct)
    speedup = openmc_result['time_s'] / yamc_result['time_s'] if yamc_result['time_s'] > 0 else float('inf')

    return {
        'detail_rows': detail_rows,
        'max_rel_diff_pct': max_rel_diff_pct,
        'rms_rel_diff_pct': rms_rel_diff_pct,
        'n_shared': n_shared,
        'n_total': n_total,
        'yamc_only_count': yamc_only_count,
        'openmc_only_count': openmc_only_count,
        'quality': quality,
        'speedup': speedup,
    }


def create_density_comparison_table(detail_rows, step_idx):
    """Create a density comparison table for a specific timestep."""
    step_rows = [dr for dr in detail_rows if dr['step'] == step_idx + 1]

    rows = []
    for dr in sorted(step_rows, key=lambda x: -max(abs(x['yamc']), abs(x['openmc']))):
        rows.append({
            'Nuclide': dr['product'],
            'YAMC': f"{dr['yamc']:.6e}",
            'OpenMC': f"{dr['openmc']:.6e}",
            'Rel. Diff (%)': f"{dr['rel_diff_pct']:.4f}%",
            'Status': 'Shared' if dr['shared'] else ('YAMC only' if abs(dr['yamc']) >= DENSITY_THRESHOLD else 'OpenMC only'),
        })

    return pd.DataFrame(rows)


def create_density_evolution_plot(detail_rows, nuclide):
    """Create density evolution plots for top products over timesteps."""

    # Get all products and their max densities
    product_max = {}
    for dr in detail_rows:
        p = dr['product']
        val = max(abs(dr['yamc']), abs(dr['openmc']))
        if p not in product_max or val > product_max[p]:
            product_max[p] = val

    # Top 10 products by max density (excluding the starting nuclide itself)
    top_products = sorted(
        [(p, v) for p, v in product_max.items() if p != nuclide],
        key=lambda x: -x[1]
    )[:10]

    if not top_products:
        return None

    steps = list(range(1, len(TRANSMUTE_TIMESTEPS) + 1))
    step_labels = [f"Step {s}\n({TRANSMUTE_PHASES[s-1]})" for s in steps]

    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('YAMC', 'OpenMC'),
        shared_yaxes=True,
        horizontal_spacing=0.05,
    )

    for idx, (product, _) in enumerate(top_products):
        yamc_densities = []
        openmc_densities = []
        for step in steps:
            ymc_val = 0.0
            omc_val = 0.0
            for dr in detail_rows:
                if dr['product'] == product and dr['step'] == step:
                    ymc_val = dr['yamc']
                    omc_val = dr['openmc']
                    break
            yamc_densities.append(ymc_val)
            openmc_densities.append(omc_val)

        color = colors[idx % len(colors)]

        fig.add_trace(
            go.Scatter(
                x=steps, y=yamc_densities, mode='lines+markers',
                name=product, line=dict(color=color),
                legendgroup=product,
            ),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=steps, y=openmc_densities, mode='lines+markers',
                name=product, line=dict(color=color, dash='dash'),
                legendgroup=product, showlegend=False,
            ),
            row=1, col=2
        )

    fig.update_xaxes(title_text="Timestep", tickvals=steps, ticktext=step_labels, row=1, col=1)
    fig.update_xaxes(title_text="Timestep", tickvals=steps, ticktext=step_labels, row=1, col=2)
    fig.update_yaxes(type="log", title_text="Atom Density (atoms/b-cm)", exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", exponentformat='e', row=1, col=2)

    fig.update_layout(
        height=500,
        title=f"{nuclide} - Product Density Evolution (Top 10)",
        legend=dict(x=1.02, y=1.0),
        margin=dict(l=60, r=150, t=60, b=60),
    )

    return fig


def create_quality_metrics_table(comparison):
    """Create a summary table of quality metrics."""
    rows = [
        {'Metric': 'Max Rel. Diff (shared)', 'Value': f"{comparison['max_rel_diff_pct']:.4f}%"},
        {'Metric': 'RMS Rel. Diff (shared)', 'Value': f"{comparison['rms_rel_diff_pct']:.4f}%"},
        {'Metric': 'Shared Products', 'Value': str(comparison['n_shared'])},
        {'Metric': 'Total Products', 'Value': str(comparison['n_total'])},
        {'Metric': 'YAMC-only entries', 'Value': str(comparison['yamc_only_count'])},
        {'Metric': 'OpenMC-only entries', 'Value': str(comparison['openmc_only_count'])},
        {'Metric': 'Quality', 'Value': comparison['quality']},
        {'Metric': 'Speedup', 'Value': f"{comparison['speedup']:.1f}x"},
    ]
    return pd.DataFrame(rows)

In [ ]:
# Discover nuclides: stable neutron nuclides that are in the chain.
# Restricted to stable until YAMC issue #45 is fixed (reduce() pulls in
# fission-yield products which triggers ~1000 cross-section downloads).
#
# When OpenMC runs, the OpenMC chain is the source of truth (OpenMC needs it
# for the deplete solve) and nuclides come from the HDF5 dir. When OpenMC is
# excluded, the OpenMC chain may be absent, so discover stable nuclides from
# the already-loaded YAMC chain (stable = present in the chain with no
# half-life entry) and source names from the local yamc Arrow cache.
import re
from pathlib import Path as _P

neutron_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron')

if RUN_OPENMC:
    import openmc.deplete

    _t0 = time.perf_counter()
    chain = openmc.deplete.Chain.from_xml(CHAIN_FILE_OPENMC)
    OPENMC_CHAIN_LOAD_S = time.perf_counter() - _t0
    print(f"OpenMC chain load: {OPENMC_CHAIN_LOAD_S:.3f}s ({len(chain.nuclides)} nuclides)")

    chain_stable_names = {nuc.name for nuc in chain.nuclides if nuc.half_life is None}
    print(f"({len(chain_stable_names)} stable)")

    h5_files = sorted(glob.glob(os.path.join(neutron_dir, '*.h5')))
    nuclides = []
    for f in h5_files:
        name = os.path.splitext(os.path.basename(f))[0]
        if '_m' in name:  # skip metastable
            continue
        if name in chain_stable_names:
            nuclides.append((name, f))

    if not nuclides:
        raise FileNotFoundError(
            f"No stable nuclides found in chain from {neutron_dir}\n"
            f"Please check that the chain file and nuclear data directory exist."
        )
else:
    chain = None
    OPENMC_CHAIN_LOAD_S = 0.0
    # Stable in the YAMC chain: present in nuclide_names with no half-life entry.
    _hl = YAMC_CHAIN.half_lives
    chain_stable_names = {
        n for n in YAMC_CHAIN.nuclide_names if '_m' not in n and _hl.get(n) is None
    }
    print(f"({len(chain_stable_names)} stable in YAMC chain)")

    yamc_cache = _P.home() / ".cache" / "yamc"
    prefix = f"{NUCLEAR_DATA}-"
    nuclides = []
    for p in sorted(yamc_cache.glob(f"{prefix}*.arrow")):
        if not p.is_dir():
            continue
        name = p.name[len(prefix):-len(".arrow")]
        if '_m' in name:  # skip metastable
            continue
        if not re.fullmatch(r"[A-Z][a-z]?\d+", name):
            continue
        if name in chain_stable_names:
            # neutron_path is nominal here; yamc reads its own arrow cache.
            nuclides.append((name, os.path.join(neutron_dir, f"{name}.h5")))

    if not nuclides:
        raise FileNotFoundError(
            f"No stable nuclides found in YAMC chain / cache at {yamc_cache}.\n"
            f"Run yamc once on any stable nuclide to populate the cache."
        )

if NUMBER_OF_SIMULATIONS:
    nuclides = nuclides[:NUMBER_OF_SIMULATIONS]

print(f"Running transmutation for {len(nuclides)} nuclides...")
for nuc, _ in nuclides:
    print(f"  {nuc}")

In [ ]:
# Run all transmutations and collect results
all_results = {}

for nuclide, neutron_path in nuclides:
    print(f"\nProcessing {nuclide}...")
    per_nuclide = {}

    if RUN_YAMC:
        try:
            print("  Running YAMC transmute...")
            # run_yamc_transmutation reduces the YAMC chain internally; the
            # third argument (an OpenMC reduced chain) is unused, so pass None
            # so the YAMC path works even when OpenMC is skipped.
            yamc_result = run_yamc_transmutation(nuclide, neutron_path, None)
            per_nuclide['yamc'] = yamc_result
            print(f"  YAMC: reduce={yamc_result['reduce_s']:.3f}s "
                  f"deplete={yamc_result['deplete_s']:.3f}s "
                  f"total={yamc_result['time_s']:.3f}s")
        except Exception as e:
            print(f"  YAMC Error: {e}")
            continue

    if RUN_OPENMC:
        try:
            # Reduce OpenMC chain (timed for symmetry with YAMC's reduce step).
            t0 = time.perf_counter()
            reduced_chain = chain.reduce([nuclide], TRANSMUTE_REDUCE_LEVEL)
            omc_reduce_s = time.perf_counter() - t0

            print("  Running OpenMC transmute...")
            openmc_result = cached_or_run(
                CACHE_DIR, NUCLEAR_DATA, "openmc", nuclide,
                lambda n=nuclide, rc=reduced_chain, rs=omc_reduce_s:
                    run_openmc_transmutation(n, rc, rs),
                meta=OPENMC_CACHE_META,
            )
            per_nuclide['openmc'] = openmc_result
            print(f"  OpenMC: reduce={openmc_result['reduce_s']:.3f}s "
                  f"deplete={openmc_result['deplete_s']:.3f}s "
                  f"total={openmc_result['time_s']:.3f}s")
        except Exception as e:
            print(f"  OpenMC Error: {e}")
            continue

    # Compare only when both backends ran.
    if 'yamc' in per_nuclide and 'openmc' in per_nuclide:
        comparison = compare_transmutation_steps(per_nuclide['yamc'], per_nuclide['openmc'])
        per_nuclide['comparison'] = comparison
        print(f"  max_rel_diff={comparison['max_rel_diff_pct']:.4f}%, "
              f"quality={comparison['quality']}, "
              f"deplete-only speedup="
              f"{per_nuclide['openmc']['deplete_s']/per_nuclide['yamc']['deplete_s']:.1f}x")

    all_results[nuclide] = per_nuclide

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")

## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- Density evolution plots (top 10 product nuclides)
- Quality metrics (max/RMS relative difference on shared products)
- Density comparison tables per timestep

In [ ]:
# Show chain-load timings once for the whole notebook
display(Markdown(
    f"**Chain load** (one-time, before any nuclide):  \n"
    f"OpenMC: {OPENMC_CHAIN_LOAD_S:.3f}s | YAMC: {YAMC_CHAIN_LOAD_S:.3f}s"
))

# Display results for each nuclide
for nuclide, results in all_results.items():
    yr = results.get('yamc')        # may be None when no yamc-* in MC_BACKENDS
    or_ = results.get('openmc')     # may be None when openmc-cpu not in MC_BACKENDS
    comparison = results.get('comparison')  # only present when both ran

    # Header
    display(Markdown(f"## {nuclide}"))

    # Performance comparison (deplete only - chain reduce is sub-millisecond)
    perf_parts = []
    if or_ is not None:
        perf_parts.append(f"OpenMC: {or_['deplete_s']:.3f}s")
    if yr is not None:
        perf_parts.append(f"YAMC: {yr['deplete_s']:.3f}s")
    if or_ is not None and yr is not None and yr['deplete_s'] > 0:
        perf_parts.append(f"Speedup: {or_['deplete_s'] / yr['deplete_s']:.1f}x")
    display(Markdown(f"**Performance:** " + " | ".join(perf_parts)))

    # Quality metrics and density tables compare OpenMC vs YAMC; without both
    # there is nothing to compare, so note it and continue rather than KeyError.
    if comparison is None:
        only = "OpenMC" if or_ is not None else "YAMC"
        display(Markdown(
            f"*Only {only} ran (MC_BACKENDS={MC_BACKENDS}); "
            f"comparison metrics and density tables are skipped.*"
        ))
        print()
        continue

    # Quality metrics table
    display(Markdown("### Quality Metrics"))
    metrics_table = create_quality_metrics_table(comparison)
    display(metrics_table.style.hide(axis='index'))

    # Density evolution plot
    fig = create_density_evolution_plot(comparison['detail_rows'], nuclide)
    if fig is not None:
        fig.show()

    # Density comparison tables per timestep
    for step_idx in range(len(TRANSMUTE_TIMESTEPS)):
        phase = TRANSMUTE_PHASES[step_idx]
        display(Markdown(f"### Step {step_idx + 1} ({phase})"))
        table = create_density_comparison_table(comparison['detail_rows'], step_idx)
        if not table.empty:
            display(table.style.hide(axis='index'))
        else:
            display(Markdown("*No products above density threshold.*"))

    print()  # Add spacing between nuclides